<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import numpy as np
import torch

# Example of Torch

### How to use (p. 6)


In [39]:
x = torch.tensor([[10., 2., 0.], [ 0., -1., 1.]])
w = torch.tensor([[1.], [0.], [1.]], requires_grad=True)
y = torch.tensor([[ 1.], [-1.]])
logit = x @ w # 행렬 곱 (logit = x * w)
margin = logit * y # 원소별 곱 (margin = logit ⊙ y)
prob = torch.sigmoid(margin) # 로지스틱/시그모이드 함수 (logistic)
log_prob = torch.log(prob) # 로그 (log) #
total_loss = -torch.mean(log_prob) # -ones/N * log_prob
print(f"total_loss = {total_loss.item():.3f}")

total_loss = 0.657


In [40]:
total_loss.backward()  # 연산 그래프를 따라 w의 기울기 자동 계산

print(f"Total Loss: {total_loss.item():.3f}\n")
print("w에 대한 기울기 (w.grad):")
print(w.grad)

Total Loss: 0.657

w에 대한 기울기 (w.grad):
tensor([[-2.2709e-04],
        [-3.6557e-01],
        [ 3.6553e-01]])


### Page 9

In [41]:
x = torch.tensor(1., requires_grad=True)
y = x ** 2
z = y ** 2
u = torch.tensor(3., requires_grad=True)
l2 = y.detach() ** 2 + u
l2.backward()

print(f"l2 = {l2.item():.3f}")
print(f"u.grad = {u.grad:.3f}")

l2 = 4.000
u.grad = 1.000


In [42]:
print(f"x.grad = {x.grad:.3f}") ### ERROR!!!

TypeError: unsupported format string passed to NoneType.__format__

In [43]:
z.backward()
print(f"x.grad = {x.grad:.3f}")

x.grad = 4.000


### Page 11

In [45]:
with torch.no_grad():
  y = x ** 2
  z = y ** 2

In [47]:
try:
  z.backward()
except RuntimeError as e:
  print(e)

element 0 of tensors does not require grad and does not have a grad_fn


### Page 13

In [ ]:
def get_training_data() -> list[Example]:
  return [
    Example(x=torch.tensor([1., 2, 0, 1]), target_y=torch.tensor([0., 1, 0])),
    Example(x=torch.tensor([-1., 0, 2, 0]), target_y=torch.tensor([1., 0, 0])),
    Example(x=torch.tensor([0., 3, 1, 0]), target_y=torch.tensor([0., 0, 1])),
  ]

def train_model(model: nn.Module,
                training_data: list[Example],
                optimizer_class=torch.optim.SGD,
                num_steps=80,
                learning_rate=0.1):
  """Train the model on `training_data`."""
  # Create data in tensor format (every row is an example)
  x = torch.stack([example.x for example in training_data])
  target_y = torch.stack([example.target_y for example in training_data])
  cross_entropy = nn.CrossEntropyLoss()
  losses: list[float] = []
  optimizer = optimizer_class(model.parameters(), lr=learning_rate)
  for step in range(num_steps):
    # Forward pass
    logits = model(x)
    loss = cross_entropy(logits, target_y)
    losses.append(loss.item())
    # Backward pass
    optimizer.zero_grad()  # Remember to do this!
    loss.backward()
    # Update parameters
    optimizer.step()
    parameters = list(model.named_parameters())
  return Chart(Data(values=[{"step": i, "loss": loss} for i, loss in enumerate(losses)])).mark_line().encode(x="step:Q", y="loss:Q").to_dict()